# Question-to-Cypher (Q2C) — E2E Evaluation

Notebook untuk mengevaluasi pipeline E2E: **Question → Cypher → Neo4j → Answer**.

**Alur:**
1. Load test data dari Google Sheets / CSV (berisi `TEST_ID`, `QUESTION`)
2. Kirim setiap pertanyaan ke backend `/api/qa`
3. Capture: `GENERATED_CYPHER`, `CYPHER_QUERY_RESULT`, `ANSWER`, `STATUS`
4. Hitung pass rate per kategori
5. Tulis hasil ke Google Sheets

**Perbedaan dari `04_evaluation.ipynb` (KG Extraction):**
- KG Extraction: query Cypher sudah ada di test data, hanya execute + compare
- Q2C E2E: **question saja** → backend generate Cypher → execute → jawab

In [49]:
# === Setup ===
import sys
import json
import os
import time
import requests
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

print(f'Project root: {PROJECT_ROOT}')

Project root: d:\TA\llm-driven-legal-kg-visualization


## Step 0: Configuration

Set `EXPERIMENT_ID`, backend URL, dan sumber test data.

**Experiment ID format:** `Q2C_{NUMBER}`

In [50]:
# === Configuration ===
EXPERIMENT_ID = "Q2C_001"              # Unique ID per eksperimen
DOCUMENT_IDS = ["POJK_11_2022", "UU_11_2008", "UU_19_2016"]          # doc_ids untuk query (list)

# Backend API
BACKEND_URL = "http://localhost:8000"   # Backend FastAPI base URL
API_TIMEOUT = 120                       # Timeout per request (seconds)
DELAY_BETWEEN_REQUESTS = 1              # Delay antar request (seconds)

# Sumber test data: 'csv' atau 'gsheets'
TEST_DATA_SOURCE = "gsheets"
CSV_PATH = "data/q2c_test_data.csv"              # Jika source = csv
GSHEETS_SHEET_NAME = "UU_11_2008_UU_19_2016_E2E_DATATEST"   # Jika source = gsheets

# Tulis hasil ke Google Sheets?
WRITE_TO_GSHEETS = True
EXPERIMENT_SHEET_NAME = f"EXP_E2E_{GSHEETS_SHEET_NAME}_{'_'.join(DOCUMENT_IDS)}_V3"    # 1 sheet per eksperimen

print(f'Experiment: {EXPERIMENT_ID}')
print(f'Document IDs: {DOCUMENT_IDS}')
print(f'Backend: {BACKEND_URL}')
print(f'Test data source: {TEST_DATA_SOURCE}')
print(f'Output sheet: {EXPERIMENT_SHEET_NAME}')

Experiment: Q2C_001
Document IDs: ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016']
Backend: http://localhost:8000
Test data source: gsheets
Output sheet: EXP_E2E_UU_11_2008_UU_19_2016_E2E_DATATEST_POJK_11_2022_UU_11_2008_UU_19_2016_V3


## Step 1: Load Test Data

In [51]:
# === Load Test Data ===
if TEST_DATA_SOURCE == "gsheets":
    from modules.google_sheets_utils import GoogleUtil
    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
    test_df = gu.load_dataframe_from_sheet(spreadsheet_id, GSHEETS_SHEET_NAME)
else:
    test_df = pd.read_csv(CSV_PATH)

print(f'Loaded {len(test_df)} test cases')
print(f'Columns: {list(test_df.columns)}')
test_df.head(10)

2026-05-18 18:09:34,426 - INFO - Retrieving worksheet 'UU_11_2008_UU_19_2016_E2E_DATATEST' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-05-18 18:09:35,915 - INFO - Successfully loaded 25 rows from worksheet 'UU_11_2008_UU_19_2016_E2E_DATATEST'.


Loaded 25 test cases
Columns: ['TEST_ID', 'QUESTION']


,TEST_ID,QUESTION
0,UU_11_2008_UU_19_2016_E2E_001,UU 11/2008 diubah oleh UU apa?
1,UU_11_2008_UU_19_2016_E2E_002,Pasal apa saja yang diubah oleh UU 19/2016?
2,UU_11_2008_UU_19_2016_E2E_003,Pasal baru apa yang disisipkan oleh UU 19/2016?
3,UU_11_2008_UU_19_2016_E2E_004,Apakah Pasal 27 sudah diamandemen?
4,UU_11_2008_UU_19_2016_E2E_005,Apakah Pasal 40 sudah diamandemen?
5,UU_11_2008_UU_19_2016_E2E_006,Apakah Pasal 45 sudah diamandemen?
6,UU_11_2008_UU_19_2016_E2E_007,Apa isi Pasal 45 menurut UU 11/2008?
7,UU_11_2008_UU_19_2016_E2E_008,Apa isi Pasal 45 menurut UU 19/2016?
8,UU_11_2008_UU_19_2016_E2E_009,Apa perbedaan sanksi pencemaran nama baik anta...
9,UU_11_2008_UU_19_2016_E2E_010,Apa yang diatur Pasal 27 menurut UU 11/2008?


## Step 2: Verify Backend is Running

In [52]:
# === Check Backend Health ===
try:
    resp = requests.get(f"{BACKEND_URL}/docs", timeout=5)
    print(f'✅ Backend is running at {BACKEND_URL} (status: {resp.status_code})')
except Exception as e:
    print(f'❌ Backend not reachable: {e}')
    print('Make sure to run: uvicorn app.main:app --reload --port 8000')

✅ Backend is running at http://localhost:8000 (status: 200)


## Step 3: Define E2E Evaluation Logic

In [53]:
def call_qa_api(question: str, doc_ids: list, backend_url: str, timeout: int = 120) -> dict:
    """Call the /api/qa endpoint and extract all response fields.
    
    Returns:
        dict with keys: status, cypher, cypher_result, answer, error, time_s,
                        references, process_steps
    """
    start = time.time()
    try:
        resp = requests.post(
            f"{backend_url}/api/qa",
            json={"question": question, "doc_ids": doc_ids},
            timeout=timeout,
        )
        elapsed = time.time() - start
        
        if resp.status_code != 200:
            return {
                'status': 'HTTP_ERROR',
                'cypher': '',
                'cypher_result': '',
                'answer': '',
                'error': f'HTTP {resp.status_code}: {resp.text[:200]}',
                'time_s': round(elapsed, 2),
                'references': '',
                'process_steps': [],
            }
        
        data = resp.json()
        steps = data.get('process_steps', [])
        answer = data.get('answer', '')
        cypher = data.get('cypher_query', '')
        
        # Extract Cypher Query Result from process_steps
        # Step 3 = Neo4j execution result (contains 'detail' and 'data')
        cypher_result_detail = ''
        cypher_result_data = []
        for s in steps:
            if s.get('step') == 3:
                cypher_result_detail = s.get('detail', '')
                cypher_result_data = s.get('data', [])
                break
        
        # Format cypher result: combine detail + raw data
        if cypher_result_data:
            cypher_result = json.dumps(cypher_result_data, ensure_ascii=False)
        else:
            cypher_result = cypher_result_detail
        
        # Determine status
        cypher_ok = any(s.get('step') == 2 and s.get('status') == 'done' for s in steps)
        has_results = 'hasil ditemukan' in cypher_result_detail and not cypher_result_detail.startswith('0')
        has_answer = bool(answer) and len(answer) > 20
        
        if cypher_ok and has_results and has_answer:
            status = 'PASS'
        elif cypher_ok and not has_results:
            status = 'FAIL'
        elif not cypher_ok:
            status = 'FAIL'
        else:
            status = 'PARTIAL'
        
        # Extract metadata
        references = '; '.join(data.get('references', []))
        
        return {
            'status': status,
            'cypher': cypher,
            'cypher_result': cypher_result,
            'cypher_result_detail': cypher_result_detail,
            'answer': answer,
            'time_s': round(elapsed, 2),
            'references': references,
            'process_steps': steps,
        }
        
    except requests.exceptions.Timeout:
        return {
            'status': 'ERROR',
            'cypher': '',
            'cypher_result': '',
            'cypher_result_detail': '',
            'answer': '',
            'error': f'Timeout ({timeout}s)',
            'time_s': timeout,
            'references': '',
            'process_steps': [],
        }
    except Exception as e:
        elapsed = time.time() - start
        return {
            'status': 'ERROR',
            'cypher': '',
            'cypher_result': '',
            'cypher_result_detail': '',
            'answer': '',
            'error': str(e),
            'time_s': round(elapsed, 2),
            'references': '',
            'process_steps': [],
        }


print('✅ E2E evaluation functions defined')

✅ E2E evaluation functions defined


## Step 4: Run E2E Evaluation

In [54]:
# === Run All Test Cases ===
results = []
total = len(test_df)

for idx, row in test_df.iterrows():
    test_id = row['TEST_ID']
    question = row['QUESTION']
    
    print(f'[{idx+1}/{total}] {test_id}: {question[:60]}...', end=' ')
    
    # Call backend API
    result = call_qa_api(
        question=question,
        doc_ids=DOCUMENT_IDS,
        backend_url=BACKEND_URL,
        timeout=API_TIMEOUT,
    )
    
    # Print status
    status_icon = {'PASS': '✅', 'FAIL': '❌', 'ERROR': '⚠️', 'PARTIAL': '🟡'}.get(result['status'], '❓')
    print(f"{status_icon} {result['status']} ({result['time_s']:.1f}s)")
    if result['cypher']:
        print(f"    Cypher: {result['cypher'][:120]}...")
    if result.get('cypher_result', ''):
        print(f"    Neo4j: {result.get('cypher_result', '')[:80]}")
    
    # Build result row (matching desired output columns)
    results.append({
        'TEST_ID': test_id,
        'QUESTION': question,
        'GENERATED_CYPHER': result['cypher'],
        'CYPHER_QUERY_RESULT': result['cypher_result'],
        'ANSWER': result['answer'],
        'STATUS': result['status'],
        'REFERENCES': result['references'],
        'TIME_S': result['time_s'],
    })
    
    # Delay between requests
    if idx < total - 1:
        time.sleep(DELAY_BETWEEN_REQUESTS)

results_df = pd.DataFrame(results)
print(f'\n=== Done: {len(results)} test cases evaluated ===')

[1/25] UU_11_2008_UU_19_2016_E2E_001: UU 11/2008 diubah oleh UU apa?... ✅ PASS (19.0s)
    Cypher: MATCH (amender:Regulasi)-[:MENGAMANDEMEN]->(original:Regulasi) WHERE original.source_document_id = 'UU_11_2008' AND amen...
    Neo4j: 1 hasil ditemukan
[2/25] UU_11_2008_UU_19_2016_E2E_002: Pasal apa saja yang diubah oleh UU 19/2016?... ✅ PASS (16.2s)
    Cypher: MATCH (p) WHERE p.source_document_id IN ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016'] AND p.source_document_id = 'UU_19_2...
    Neo4j: 13 hasil ditemukan
[3/25] UU_11_2008_UU_19_2016_E2E_003: Pasal baru apa yang disisipkan oleh UU 19/2016?... ✅ PASS (13.1s)
    Cypher: MATCH (p:Pasal) WHERE p.source_document_id = 'UU_19_2016' AND p.jenis_perubahan = 'menyisipkan' RETURN p.label AS pasal,...
    Neo4j: 3 hasil ditemukan
[4/25] UU_11_2008_UU_19_2016_E2E_004: Apakah Pasal 27 sudah diamandemen?... ✅ PASS (15.8s)
    Cypher: MATCH (r1:Regulasi {source_document_id:'UU_11_2008'})<-[:MENGAMANDEMEN]-(r2:Regulasi) WHERE r2.source_document

## Step 5: Summary

In [55]:
# === Overall Summary ===
total = len(results_df)
passed = len(results_df[results_df['STATUS'] == 'PASS'])
failed = len(results_df[results_df['STATUS'] == 'FAIL'])
errors = len(results_df[results_df['STATUS'] == 'ERROR'])
partial = len(results_df[results_df['STATUS'] == 'PARTIAL'])
avg_time = results_df['TIME_S'].mean()

print(f'╔══════════════════════════════════════════╗')
print(f'║  Q2C E2E Evaluation Report               ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Experiment: {EXPERIMENT_ID:<27s} ║')
print(f'║  Documents:  {", ".join(DOCUMENT_IDS):<27s} ║')
print(f'║  Backend:    {BACKEND_URL:<27s} ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Total:   {total:>3d}                              ║')
print(f'║  Pass:    {passed:>3d}  ({passed/total:.1%})                     ║')
print(f'║  Fail:    {failed:>3d}  ({failed/total:.1%})                     ║')
print(f'║  Error:   {errors:>3d}                              ║')
print(f'║  Partial: {partial:>3d}                              ║')
print(f'║  Avg Time: {avg_time:.1f}s                          ║')
print(f'╚══════════════════════════════════════════╝')

╔══════════════════════════════════════════╗
║  Q2C E2E Evaluation Report               ║
╠══════════════════════════════════════════╣
║  Experiment: Q2C_001                     ║
║  Documents:  POJK_11_2022, UU_11_2008, UU_19_2016 ║
║  Backend:    http://localhost:8000       ║
╠══════════════════════════════════════════╣
║  Total:    25                              ║
║  Pass:     21  (84.0%)                     ║
║  Fail:      4  (16.0%)                     ║
║  Error:     0                              ║
║  Partial:   0                              ║
║  Avg Time: 14.9s                          ║
╚══════════════════════════════════════════╝


In [56]:
# === Failed/Error Test Cases ===
failed_df = results_df[results_df['STATUS'] != 'PASS']

if len(failed_df) == 0:
    print('🎉 All test cases passed!')
else:
    print(f'\n❌ Failed/Error test cases ({len(failed_df)}):\n')
    for _, row in failed_df.iterrows():
        print(f"  {row['TEST_ID']}: {row['QUESTION'][:60]}")
        print(f"    Status: {row['STATUS']}")
        if row['GENERATED_CYPHER']:
            print(f"    Cypher: {row['GENERATED_CYPHER'][:100]}...")
        print()


❌ Failed/Error test cases (4):

  UU_11_2008_UU_19_2016_E2E_018: Apa saja pasal yang berisi sanksi di UU 19/2016?
    Status: FAIL
    Cypher: MATCH (node)-[:MENETAPKAN_SANKSI]->(s:Sanksi) WHERE (labels(node) CONTAINS 'Ayat' OR labels(node) CO...

  UU_11_2008_UU_19_2016_E2E_019: Apakah ada pasal di UU 11/2008 yang tidak berubah?
    Status: FAIL
    Cypher: MATCH (p_original:Pasal)
WHERE p_original.source_document_id = 'UU_11_2008'
OPTIONAL MATCH (p_amende...

  UU_11_2008_UU_19_2016_E2E_020: Apa definisi Penyelenggara Sistem Elektronik yang ditambahka
    Status: FAIL
    Cypher: MATCH (p)-[:MENDEFINISIKAN]->(k:KonsepHukum) WHERE toLower(k.label) CONTAINS 'penyelenggara sistem e...

  UU_11_2008_UU_19_2016_E2E_022: Apa hubungan antara UU 11/2008 dan UU 19/2016?
    Status: FAIL
    Cypher: MATCH (r1:Regulasi {source_document_id: 'UU_19_2016'})-[:MENGAMANDEMEN]->(r2:Regulasi {source_docume...



## Step 6: Save Results

In [57]:
# === Save to CSV ===
output_dir = 'data/evaluation'
os.makedirs(output_dir, exist_ok=True)

output_csv = f'{output_dir}/q2c_eval_{EXPERIMENT_ID}.csv'
results_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f'✅ Results saved to: {output_csv}')

✅ Results saved to: data/evaluation/q2c_eval_Q2C_001.csv


In [58]:
# === (Optional) Write to Google Sheets ===
if WRITE_TO_GSHEETS:
    from modules.google_sheets_utils import GoogleUtil, GoogleSheetsWriter
    import gspread

    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')

    # Auto-create worksheet if it does not exist
    try:
        info = gu._get_google_info(gu.private_key, gu.client_email)
        client = gu._get_client(info, gu.GOOGLE_SHEETS_SCOPES)
        sh = client.open_by_key(spreadsheet_id)
        try:
            ws = sh.worksheet(EXPERIMENT_SHEET_NAME)
            print(f'ℹ️  Sheet "{EXPERIMENT_SHEET_NAME}" already exists.')
        except gspread.exceptions.WorksheetNotFound:
            ws = sh.add_worksheet(title=EXPERIMENT_SHEET_NAME, rows=1000, cols=20)
            # Write header row
            headers = list(results_df.columns)
            ws.update([headers], "A1")
            print(f'✅ Created new sheet: "{EXPERIMENT_SHEET_NAME}"')
    except Exception as e:
        print(f'⚠️  Could not auto-create sheet: {e}')

    writer = GoogleSheetsWriter(
        google_util=gu,
        sheet_id=spreadsheet_id,
        worksheet_name=EXPERIMENT_SHEET_NAME,
        batch_size=5,
    )

    result = writer.write_dataframe(results_df)
    print(f'✅ Written to Google Sheets: {EXPERIMENT_SHEET_NAME}')
    print(f'   Success: {result.successful_rows}, Failed: {result.failed_rows}')
else:
    print('ℹ️  WRITE_TO_GSHEETS = False, skipping Google Sheets upload.')
    print(f'   Set WRITE_TO_GSHEETS = True to write results to sheet "{EXPERIMENT_SHEET_NAME}"')


✅ Created new sheet: "EXP_E2E_UU_11_2008_UU_19_2016_E2E_DATATEST_POJK_11_2022_UU_11_2008_UU_19_2016_V3"


  0%|          | 0/5 [00:00<?, ?it/s]2026-05-18 18:16:19,613 - INFO - Successfully wrote row 1/25
2026-05-18 18:16:22,654 - INFO - Successfully wrote row 2/25
2026-05-18 18:16:24,865 - INFO - Successfully wrote row 3/25
2026-05-18 18:16:26,621 - INFO - Successfully wrote row 4/25
2026-05-18 18:16:28,647 - INFO - Successfully wrote row 5/25
 20%|██        | 1/5 [00:12<00:51, 12.95s/it]2026-05-18 18:16:32,290 - INFO - Successfully wrote row 6/25
2026-05-18 18:16:34,324 - INFO - Successfully wrote row 7/25
2026-05-18 18:16:35,971 - INFO - Successfully wrote row 8/25
2026-05-18 18:16:37,622 - INFO - Successfully wrote row 9/25
2026-05-18 18:16:39,806 - INFO - Successfully wrote row 10/25
 40%|████      | 2/5 [00:24<00:35, 11.90s/it]2026-05-18 18:16:43,811 - INFO - Successfully wrote row 11/25
2026-05-18 18:16:45,558 - INFO - Successfully wrote row 12/25
2026-05-18 18:16:47,295 - INFO - Successfully wrote row 13/25
2026-05-18 18:16:49,034 - INFO - Successfully wrote row 14/25
2026-05-18 18:

✅ Written to Google Sheets: EXP_E2E_UU_11_2008_UU_19_2016_E2E_DATATEST_POJK_11_2022_UU_11_2008_UU_19_2016_V3
   Success: 25, Failed: 0


In [59]:
# === Preview Results ===
print(f"\n{'='*80}")
print(f"RESULTS PREVIEW — First 5 rows")
print(f"{'='*80}")

for _, row in results_df.head(5).iterrows():
    print(f"\n--- {row['TEST_ID']} [{row['STATUS']}] ---")
    print(f"Q: {row['QUESTION']}")
    print(f"Cypher: {row['GENERATED_CYPHER'][:150]}..." if len(str(row['GENERATED_CYPHER'])) > 150 else f"Cypher: {row['GENERATED_CYPHER']}")
    print(f"Neo4j Result: {str(row['CYPHER_QUERY_RESULT'])[:200]}..." if len(str(row['CYPHER_QUERY_RESULT'])) > 200 else f"Neo4j Result: {row['CYPHER_QUERY_RESULT']}")
    print(f"Answer: {str(row['ANSWER'])[:200]}..." if len(str(row['ANSWER'])) > 200 else f"Answer: {row['ANSWER']}")


RESULTS PREVIEW — First 5 rows

--- UU_11_2008_UU_19_2016_E2E_001 [PASS] ---
Q: UU 11/2008 diubah oleh UU apa?
Cypher: MATCH (amender:Regulasi)-[:MENGAMANDEMEN]->(original:Regulasi) WHERE original.source_document_id = 'UU_11_2008' AND amender.source_document_id IN ['PO...
Neo4j Result: 1 hasil ditemukan
Answer: Undang-Undang Nomor 11 Tahun 2008 tentang Informasi dan Transaksi Elektronik (UU ITE) telah diubah oleh Undang-Undang Nomor 19 Tahun 2016.

**Menurut UU No. 19 Tahun 2016 tentang Perubahan atas Undang...

--- UU_11_2008_UU_19_2016_E2E_002 [PASS] ---
Q: Pasal apa saja yang diubah oleh UU 19/2016?
Cypher: MATCH (p) WHERE p.source_document_id IN ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016'] AND p.source_document_id = 'UU_19_2016' AND p.jenis_perubahan IS ...
Neo4j Result: 13 hasil ditemukan
Answer: Berdasarkan data yang diberikan, UU No. 19 Tahun 2016 melakukan perubahan terhadap pasal-pasal berikut:

**Menurut UU No. 19 Tahun 2016 tentang Perubahan atas UU ITE:**
*   **Pasal 1** (

In [60]:
print('\n✅ Evaluation complete.')


✅ Evaluation complete.
